# Reports and replay

**A test result is only worth keeping if you can gate a build on it, save it, and rerun it months later on another machine.**

`Pipeline.test` returns a `TestReport`, and this chapter builds it: a finding for each check, a record for each case, and a note of how the run was made (the model settings, the check settings, the library versions and a fingerprint of the TimeGym source). All of it is plain JSON.

Below we read a report, gate on it the way CI would, save it and load it back, see which models replay can rebuild, and replay a case from the saved file. Along the way we pin down what a PASS does and does not establish.

**Development notebook.** Expand the collapsed cells to edit the code, then restart the kernel and Run All ([build workflow](README.md#develop-the-package-here)).

In [1]:
#| default_exp reporting

In [2]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)

## What a report holds

A report has three layers: a **finding** for each check on each forecast, a **case** for each scenario and seed, and the **report** for the whole run. These cells write `timegym/reporting.py`.

In [3]:
#| export reporting
from __future__ import annotations

from dataclasses import dataclass, field
from timegym._internal import _json_safe, nonnegative_int
from enum import Enum
import json
from pathlib import Path
from collections import Counter
import math

import numpy as np

### Five verdicts

Every finding ends in one of five verdicts (`DiagnosticStatus`):

| Verdict | Meaning |
|---|---|
| PASS | passed |
| FAIL | broken output |
| ERROR | crashed |
| INCONCLUSIVE | scored, but no pass mark was set |
| UNSUPPORTED | not checked: the forecast lacks what the check needs |

Only FAIL and ERROR count as failures. A second label, the category (`DiagnosticCategory`), says what kind of evidence a finding is: an output check, a score, a statistical test against the answer key, a promise the model makes, a crash, and so on.

In [4]:
#| export reporting
class DiagnosticStatus(str, Enum):
    """The five outcomes a diagnostic can report.

    ``PASS`` and ``FAIL`` are verdicts of an implemented check. ``INCONCLUSIVE``
    marks evidence without a verdict, ``UNSUPPORTED`` a check that does not
    apply and ``ERROR`` an execution failure. Members equal their string values.
    """

    PASS = "PASS"
    FAIL = "FAIL"
    INCONCLUSIVE = "INCONCLUSIVE"
    UNSUPPORTED = "UNSUPPORTED"
    ERROR = "ERROR"

class DiagnosticCategory(str, Enum):
    """The kind of evidence a diagnostic records; members equal their string values.

    ``correctness`` is a forecast contract, ``evaluation`` the evaluator's own
    inputs, ``capability`` and ``uncertainty`` descriptive scores,
    ``distribution`` a statistical check against a reference law,
    ``metamorphic`` a declared relation and ``execution`` a model exception.
    """

    CORRECTNESS = "correctness"
    EVALUATION = "evaluation"
    CAPABILITY = "capability"
    UNCERTAINTY = "uncertainty"
    DISTRIBUTION = "distribution"
    METAMORPHIC = "metamorphic"
    EXECUTION = "execution"


REPORT_SCHEMA_VERSION = "2.0"


### One finding

A `DiagnosticResult` holds a check's name, its verdict and category, one sentence for people, and the evidence behind it. The evidence is converted to plain JSON as it comes in, and NaN or infinity is refused right there, so a finding can always be saved.

In [5]:
#| export reporting
@dataclass
class DiagnosticResult:
    """One named finding: its status, category, message and evidence.

    ``status`` and ``category`` are checked against ``DiagnosticStatus`` and
    ``DiagnosticCategory`` and stored as strings. ``evidence`` is copied into
    JSON-safe values.
    """

    name: str
    status: str
    category: str
    message: str
    evidence: dict = field(default_factory=dict)

    def __post_init__(self):
        _report_text(self.name, "diagnostic.name")
        _report_text(self.message, "diagnostic.message", allow_empty=True)
        try:
            self.status = DiagnosticStatus(self.status).value
        except (TypeError, ValueError) as exc:
            raise ValueError("Unsupported diagnostic status") from exc
        try:
            self.category = DiagnosticCategory(self.category).value
        except (TypeError, ValueError) as exc:
            raise ValueError("Unsupported diagnostic category") from exc
        if not isinstance(self.evidence, dict):
            raise ValueError("diagnostic.evidence must be an object")
        self.name, self.message = str(self.name), str(self.message)
        self.evidence = _json_safe(self.evidence)

    @property
    def failed(self) -> bool:
        """Whether this finding counts as a detected failure in CI and plots."""
        return self.status in (DiagnosticStatus.FAIL, DiagnosticStatus.ERROR)

    def to_dict(self):
        """Revalidate the finding and return it as a JSON-safe dictionary."""
        self.__post_init__()
        return _json_safe(self)

### One case

A `CaseResult` holds the findings and scores for one scenario and seed, plus the case's manifest: the recipe that replay uses to regenerate the same series. Every score must be a finite number.

In [6]:
#| export reporting
@dataclass
class CaseResult:
    """The diagnostics and scores recorded for one case.

    ``scenario`` labels the case and ``seed`` is its generator seed.
    ``metrics`` maps score names to finite numbers, and ``manifest`` is the
    case manifest that replay rebuilds a generated scenario from.
    """

    scenario: str
    seed: int | None
    diagnostics: list[DiagnosticResult] = field(default_factory=list)
    metrics: dict = field(default_factory=dict)
    manifest: dict = field(default_factory=dict)

    def __post_init__(self):
        _report_text(self.scenario, "case.scenario")
        self.scenario = str(self.scenario)
        if self.seed is not None:
            self.seed = nonnegative_int(self.seed, "case.seed")
        if not isinstance(self.diagnostics, list):
            raise ValueError("case.diagnostics must be a list")
        for diagnostic in self.diagnostics:
            if not isinstance(diagnostic, DiagnosticResult):
                raise ValueError("case.diagnostics must contain DiagnosticResult objects")
            diagnostic.__post_init__()
        if not isinstance(self.metrics, dict) or not isinstance(self.manifest, dict):
            raise ValueError("case.metrics and manifest must be objects")
        for name, value in self.metrics.items():
            _report_text(name, "case.metrics key")
            if (isinstance(value, (bool, np.bool_))
                    or not isinstance(value, (int, float, np.integer, np.floating))
                    or (isinstance(value, (float, np.floating)) and not math.isfinite(value))):
                raise ValueError(f"case.metrics.{name} must be a finite number")
        self.metrics = _json_safe(self.metrics)
        self.manifest = _json_safe(self.manifest)

    def to_dict(self):
        """Revalidate the case and return it as a JSON-safe dictionary."""
        self.__post_init__()
        return _json_safe(self)

### Reading saved data strictly

A saved report comes from outside the program, so loading copies it field by field and refuses anything that doesn't look exactly like a saved report: a missing or extra field, NaN or infinity, a key that isn't text.

In [7]:
#| export reporting
def _report_json_value(value, path="report"):
    """Copy a JSON value while rejecting nonfinite numbers and coercions."""
    if value is None or type(value) in (str, bool, int):
        return value
    if type(value) is float:
        if not math.isfinite(value):
            raise ValueError(f"{path} contains a nonfinite number")
        return value
    if isinstance(value, list):
        return [_report_json_value(item, f"{path}[{index}]") for index, item in enumerate(value)]
    if isinstance(value, dict):
        if any(not isinstance(key, str) for key in value):
            raise ValueError(f"{path} must use string object keys")
        return {key: _report_json_value(item, f"{path}.{key}") for key, item in value.items()}
    raise ValueError(f"{path} contains a non-JSON value of type {type(value).__name__}")


def _report_fields(value, expected, path):
    if not isinstance(value, dict) or set(value) != set(expected):
        raise ValueError(f"{path} must contain exactly these fields: {', '.join(sorted(expected))}")


def _report_text(value, path, *, allow_empty=False):
    if not isinstance(value, str) or (not allow_empty and not value):
        raise ValueError(f"{path} must be a {'possibly empty' if allow_empty else 'nonempty'} string")

### Output checks in plain words

Before scoring, every forecast goes through the output checks: did the model run, are the dates and the series the requested ones, is the shape right, are the values finite, are the quantiles in order? Each check groups the findings that can fail it, so a report can say in plain words which checks passed, and in how many cases. A check that didn't run, such as quantile order for a forecast without quantiles, is left out rather than counted as passed.

In [8]:
#| export reporting
# The output checks in plain words, in display order. Each forecast is
# checked before it is scored; a case passes a check when the check ran and
# none of the diagnostics it covers failed.
CHECKS = {
    "runs": "Runs without errors",
    "dates": "Forecasts the requested dates",
    "series": "Returns every series, in order",
    "shape": "Right number of steps and series",
    "finite": "No missing or infinite values",
    "quantiles": "Quantiles in order",
    "dependence": "Joint spread matches the answer key",
}
_CHECK_CODES = {
    "series": {"target_names", "target_mismatch"},
    "shape": {"point_shape", "quantile_shape", "sample_shape", "missing_prediction"},
    "quantiles": {"invalid_levels", "duplicate_levels", "nonmonotonic_levels",
                  "missing_quantiles", "quantile_crossing"},
}


def _check_key(name):
    """Map a diagnostic name to the plain check it belongs to, if any."""
    if name in ("model_execution", "forecast_contract"):
        return "runs" if name == "model_execution" else None
    if name == "reference_covariance":
        return "dependence"
    if not name.startswith("contract_"):
        return None
    code = name[len("contract_"):]
    if code.startswith("index_"):
        return "dates"
    if code.startswith(("nonfinite_", "nonreal_")):
        return "finite"
    return next((key for key, codes in _CHECK_CODES.items() if code in codes), None)


def _case_checks(case):
    """Return ``{check: True/False}`` for the checks that ran on one case."""
    by_name = {}
    for diagnostic in case.diagnostics:
        by_name.setdefault(diagnostic.name, []).append(diagnostic)
    crashed = any(d.failed for name in ("model_execution", "forecast_contract")
                  for d in by_name.get(name, []))
    outcome = {"runs": not crashed}
    if crashed:
        return outcome
    failed = {_check_key(d.name) for d in case.diagnostics if d.failed}
    for key in ("dates", "series", "shape", "finite"):
        outcome[key] = key not in failed
    quantiles_given = any(d.status != "UNSUPPORTED" for d in by_name.get("quantile_accuracy", []))
    if quantiles_given or "quantiles" in failed:
        outcome["quantiles"] = "quantiles" not in failed
    if any(d.status in ("PASS", "FAIL") for d in by_name.get("reference_covariance", [])):
        outcome["dependence"] = "dependence" not in failed
    return outcome

### The report

`TestReport` holds every case and the run's metadata. `summary` counts the verdicts, `checks` counts passes and failures per output check, `assert_passed()` is the gate for CI, `to_json` and `read` save and load the report, and `replay_case` reruns one case.

In [9]:
#| export reporting
@dataclass
class TestReport:
    """Diagnostics and scores for every case of one run, with run metadata.

    A PASS is scoped to a check; unsupported checks do not establish success.
    """

    __test__ = False  # Prevent pytest from trying to collect the dataclass.

    cases: list[CaseResult]
    metadata: dict = field(default_factory=dict)

    def __post_init__(self):
        if not isinstance(self.cases, list) or not isinstance(self.metadata, dict):
            raise ValueError("report.cases must be a list and report.metadata must be an object")
        for case in self.cases:
            if not isinstance(case, CaseResult):
                raise ValueError("report.cases must contain CaseResult objects")
            case.__post_init__()
        self.metadata = _json_safe(self.metadata)

    @property
    def summary(self):
        """Count cases, diagnostics by status, and failed cases.

        A case fails when any of its diagnostics is ``FAIL`` or ``ERROR``.
        """
        counts = Counter(diagnostic.status for case in self.cases for diagnostic in case.diagnostics)
        return {
            "cases": len(self.cases),
            "diagnostics": sum(counts.values()),
            "status_counts": {status.value: counts[status.value] for status in DiagnosticStatus},
            "failed_cases": sum(any(d.failed for d in case.diagnostics) for case in self.cases),
        }

    def to_dict(self):
        """Revalidate the report and return it with its schema version and summary."""
        self.__post_init__()
        return {"schema_version": REPORT_SCHEMA_VERSION, "cases": _json_safe(self.cases), "metadata": _json_safe(self.metadata), "summary": self.summary}

    def to_json(self, path=None):
        """Return the report as indented JSON, also writing it to ``path`` when given."""
        serialized = json.dumps(self.to_dict(), indent=2, sort_keys=True, allow_nan=False)
        if path is not None:
            Path(path).write_text(serialized + "\n", encoding="utf-8")
        return serialized


    @classmethod
    def from_dict(cls, payload):
        """Load schema 2.0 data without importing or constructing saved models.

        Unknown fields/statuses, malformed records, nonfinite metrics, and a
        saved summary inconsistent with the diagnostics are rejected. Metadata
        and evidence remain JSON data. The returned report owns copied values.
        """
        payload = _report_json_value(payload)
        _report_fields(payload, {"schema_version", "cases", "metadata", "summary"}, "report")
        if payload["schema_version"] != REPORT_SCHEMA_VERSION:
            raise ValueError(f"Unsupported report schema_version; expected {REPORT_SCHEMA_VERSION!r}")
        if not isinstance(payload["cases"], list) or not isinstance(payload["metadata"], dict):
            raise ValueError("report.cases must be a list and report.metadata must be an object")
        cases = []
        for index, record in enumerate(payload["cases"]):
            path = f"report.cases[{index}]"
            _report_fields(record, {"scenario", "seed", "diagnostics", "metrics", "manifest"}, path)
            if not isinstance(record["diagnostics"], list):
                raise ValueError(f"{path}.diagnostics must be a list")
            diagnostics = []
            for diagnostic_index, item in enumerate(record["diagnostics"]):
                diagnostic_path = f"{path}.diagnostics[{diagnostic_index}]"
                _report_fields(item, {"name", "status", "category", "message", "evidence"}, diagnostic_path)
                diagnostics.append(DiagnosticResult(**item))
            cases.append(CaseResult(record["scenario"], record["seed"], diagnostics,
                                    record["metrics"], record["manifest"]))
        report = cls(cases=cases, metadata=payload["metadata"])
        # JSON comparison also distinguishes true from 1 and 1.0 from 1.
        if json.dumps(payload["summary"], sort_keys=True) != json.dumps(report.summary, sort_keys=True):
            raise ValueError("Stored report.summary disagrees with the derived diagnostic counts")
        return report

    @classmethod
    def from_json(cls, text):
        """Parse JSON text/bytes, rejecting duplicate keys and nonfinite numbers.

        This method never interprets a string as a path. Use ``read(path)``
        for a saved report. Loading data does not execute replay or model code.
        """
        if not isinstance(text, (str, bytes, bytearray)):
            raise TypeError("from_json expects JSON text or bytes; use read(path) for a file")

        def unique_object(pairs):
            result = {}
            for key, value in pairs:
                if key in result:
                    raise ValueError(f"Duplicate JSON object key: {key!r}")
                result[key] = value
            return result

        def invalid_constant(value):
            raise ValueError(f"Nonfinite JSON constant is not permitted: {value}")

        try:
            payload = json.loads(text, object_pairs_hook=unique_object, parse_constant=invalid_constant)
        except (ValueError, UnicodeError) as exc:
            raise ValueError(f"Invalid report JSON: {exc}") from exc
        return cls.from_dict(payload)

    @classmethod
    def read(cls, path):
        """Read and validate a report file; paths are explicit, never guessed."""
        return cls.from_json(Path(path).read_bytes())

    @property
    def checks(self):
        """Count, per plain-language output check, the cases that passed and failed.

        Returns one dictionary per check that ran on at least one case, in the
        order of ``CHECKS``: ``{"check", "title", "passed", "failed"}``.
        Scores are not checks: they have no pass mark.
        """
        counts = {key: [0, 0] for key in CHECKS}
        for case in self.cases:
            for key, passed in _case_checks(case).items():
                counts[key][0 if passed else 1] += 1
        return [{"check": key, "title": CHECKS[key], "passed": passed, "failed": failed}
                for key, (passed, failed) in counts.items() if passed + failed]

    def assert_passed(self):
        """Raise when an output check failed or the model raised; otherwise return self.

        The error names each failed check in plain words, then lists every
        failing case. Scores never fail this gate: TimeGym sets no accuracy
        pass mark, so INCONCLUSIVE and UNSUPPORTED results do not raise.
        """
        failures = [f"  {case.scenario}[seed={case.seed}] {diagnostic.name}: {diagnostic.message}"
                    for case in self.cases for diagnostic in case.diagnostics
                    if diagnostic.failed]
        if failures:
            failed = [f"{row['title']}: failed in {row['failed']} of {row['passed'] + row['failed']} cases"
                      for row in self.checks if row["failed"]]
            raise AssertionError("TimeGym detected failures:\n" + "\n".join(failed + failures))
        return self

    def replay_case(self, case_index=0):
        """Re-evaluate one case when its built-in model configuration is replayable."""
        from timegym.pipeline import replay_case
        return replay_case(self, case_index)

## How the run was made

To rerun a case later you need more than its seed: the model settings, the check settings and the environment the run used. `Pipeline.test` writes all three into the report's metadata with the functions below. These cells write `timegym/provenance.py`.

In [10]:
#| export provenance
from __future__ import annotations
from timegym._internal import positive_int, nonnegative_int

import hashlib
from importlib import metadata
import inspect
from pathlib import Path
import platform

import numpy as np

from timegym import models
from timegym.models import REDUCTIONS, GaussianAR1Forecaster, NaiveForecaster
from timegym.target import transform_config, transform_from_config
from timegym._internal import _json_safe


_BUILTINS = {"NaiveForecaster": NaiveForecaster, "GaussianAR1Forecaster": GaussianAR1Forecaster}
PIPELINE_CONFIG_SCHEMA_VERSION = 3


### Naming a model without rebuilding it

A model that replay can't rebuild is still named in the report: its module, its class and, for scikit-learn estimators, its parameters.

In [11]:
#| export provenance
def _descriptor(value):
    selected = value if inspect.isclass(value) or inspect.isfunction(value) else type(value)
    return {"module": getattr(selected, "__module__", None), "qualname": getattr(selected, "__qualname__", type(value).__name__)}

In [12]:
#| export provenance
def _describe_estimator(estimator):
    description = _descriptor(estimator)
    if not inspect.isclass(estimator) and callable(getattr(estimator, "get_params", None)):
        try:
            description["parameters"] = _json_safe(estimator.get_params(deep=True))
        except Exception as exc:
            description["parameter_inspection_error"] = {"type": type(exc).__name__, "message": str(exc)}
    return description

### Model settings, and whether replay may rebuild them

`pipeline_config` marks a run replayable only when the model is one of TimeGym's built-in models (the exact class, not a subclass that shares its name) and any target transform is a built-in one too. The AR(1) baseline draws random samples, so it also needs a fixed seed. Every other model gets a reason instead.

In [13]:
#| export provenance
def pipeline_config(pipeline):
    """Describe configuration without constructing a factory or fitting a model."""
    replayable = False
    if pipeline.steps:
        model = {"kind": "sklearn_steps", "steps": [_describe_estimator(step) for step in pipeline.steps],
                 "reason": "Arbitrary estimators are not reconstructed from report data."}
    elif pipeline.model_factory is not None:
        model = {"kind": "factory", **_descriptor(pipeline.model_factory),
                 "reason": "Factory execution requires explicitly supplied user code."}
    else:
        supplied = pipeline.model
        model_class = NaiveForecaster if supplied is None else (supplied if inspect.isclass(supplied) else type(supplied))
        if model_class in _BUILTINS.values():
            # Only exact bundled classes are constructed. Subclasses do not
            # inherit replay permission merely by sharing a class name.
            template = model_class() if supplied is None or inspect.isclass(supplied) else supplied
            parameters = template.params()
            model = {"kind": "builtin", "class": model_class.__name__, "parameters": _json_safe(parameters)}
            replayable = True
            if model_class is GaussianAR1Forecaster and template.seed is None:
                replayable = False
                model["reason"] = "Exact replay requires a fixed nonnegative integer model seed."
        else:
            model = {"kind": "external", **_describe_estimator(supplied),
                     "reason": "Only exact bundled native baseline classes support automatic replay."}
    transform = None
    if pipeline.target_transform is not None:
        transform = transform_config(pipeline.target_transform)
        replayable = replayable and transform["kind"] == "builtin"
    return _json_safe({"schema_version": PIPELINE_CONFIG_SCHEMA_VERSION, "model": model, "window_length": pipeline.window_length,
                       "reduction": pipeline.reduction, "target_transform": transform, "replayable": replayable})

### Rebuilding a model from saved settings

`pipeline_from_config` goes the other way. It accepts only the model names on a short list, with exactly the parameters their constructors take, and it never imports a module named in a file.

In [14]:
#| export provenance
def pipeline_from_config(config):
    """Validate configuration and return allowlisted Pipeline constructor arguments."""
    if (not isinstance(config, dict) or type(config.get("schema_version")) is not int
            or config["schema_version"] not in (2, PIPELINE_CONFIG_SCHEMA_VERSION)):
        raise ValueError("Unsupported or incomplete pipeline configuration schema")
    expected = {"schema_version", "model", "window_length", "reduction", "replayable"}
    if config["schema_version"] == PIPELINE_CONFIG_SCHEMA_VERSION:
        expected.add("target_transform")
    if set(config) != expected:
        raise ValueError("Unsupported or incomplete pipeline configuration schema")
    if config["reduction"] not in REDUCTIONS:
        raise ValueError(f"Unknown reduction {config['reduction']!r}; choose from {REDUCTIONS}")
    if config["replayable"] is not True:
        raise ValueError("This model configuration is not automatically replayable; supply its original model code")
    model = config["model"]
    if not isinstance(model, dict) or set(model) != {"kind", "class", "parameters"} or model["kind"] != "builtin":
        raise ValueError("Replay accepts only allowlisted built-in model descriptions")
    name = model["class"]
    if not isinstance(name, str) or name not in _BUILTINS:
        raise ValueError("Model class is not in the replay allowlist")
    parameters = model["parameters"]
    defaults = _BUILTINS[name]().params()
    # Configurations written before configurable AR(1) quantiles used the default levels.
    if name == "GaussianAR1Forecaster" and isinstance(parameters, dict) and "levels" not in parameters:
        parameters = {**parameters, "levels": defaults["levels"]}
    if not isinstance(parameters, dict) or set(parameters) != set(defaults):
        raise ValueError("Unexpected or missing built-in model constructor parameters")
    window = config["window_length"]
    window = positive_int(window, 'window_length')
    restored = _BUILTINS[name](**parameters)
    if isinstance(restored, GaussianAR1Forecaster) and restored.seed is None:
        raise ValueError("Exact replay requires a fixed nonnegative integer model seed")
    transform = config.get("target_transform")
    return {"model": restored, "window_length": window, "reduction": config["reduction"],
            "target_transform": None if transform is None else transform_from_config(transform)}

### Versions and a source fingerprint

`runtime_provenance` records the versions of Python, TimeGym, NumPy, pandas, SciPy and scikit-learn, and a SHA-256 hash of every Python file in the installed `timegym` package. Two reports with the same hash ran the same TimeGym code.

In [15]:
#| export provenance
def runtime_provenance():
    """Package/dependency versions and a content digest of available Python source."""
    versions = {}
    for name in ("timegym", "numpy", "pandas", "scipy", "scikit-learn"):
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    root = Path(models.__file__).resolve().parent
    digest = hashlib.sha256()
    paths = sorted(root.rglob("*.py"))
    for path in paths:
        digest.update(path.relative_to(root).as_posix().encode("utf-8") + b"\0")
        digest.update(path.read_bytes() + b"\0")
    return {"timegym_version": versions["timegym"], "python": platform.python_version(),
            "python_implementation": platform.python_implementation(), "dependencies": versions,
            "source": {"algorithm": "sha256", "scope": "timegym/**/*.py relative paths and contents",
                       "digest": digest.hexdigest(), "file_count": len(paths)}}

### Check settings

`diagnostic_settings` validates the settings that can change a verdict: whether to run the joint-spread check (`check_dependence`), and that check's calibration draws, error rate and seed offset. They are saved with the report, so replay reruns the same checks.

In [16]:
#| export provenance
def diagnostic_settings(*, check_dependence=False, n_calibration=99, alpha=0.05, dependence_seed_offset=100_000):
    """Validate all settings that influence Pipeline.test evaluation."""
    if not isinstance(check_dependence, bool):
        raise ValueError("check_dependence must be a boolean")
    n_calibration = positive_int(n_calibration, 'n_calibration')
    dependence_seed_offset = nonnegative_int(dependence_seed_offset, 'dependence_seed_offset')
    if isinstance(alpha, bool) or not isinstance(alpha, (int, float)) or not np.isfinite(alpha) or not 0 < alpha < 1:
        raise ValueError("alpha must be finite and in (0, 1)")
    return {"check_dependence": check_dependence, "n_calibration": n_calibration,
            "alpha": float(alpha), "dependence_seed_offset": dependence_seed_offset}

## Read a report

The examples run this chapter's code. `Pipeline.test` comes from the installed package and returns the installed `TestReport`, so `local()` reads each report back through the `TestReport` defined above.

In [17]:
from timegym import GaussianAR1Forecaster, NaiveForecaster, Pipeline, Scenario
from timegym import TestReport as InstalledTestReport

# The examples use this chapter's definitions, not the installed copies.
assert TestReport.__module__ == __name__
assert pipeline_config.__module__ == runtime_provenance.__module__ == __name__


def local(report):
    """Read a report back through this chapter's TestReport."""
    return TestReport.from_json(report.to_json())

We test the last-value forecaster on two kinds of series, mean reversion (AR(1)) and noise around a repeating cycle, with two draws of each. `summary` counts the verdicts and `checks` lists each output check that ran:

In [18]:
suite = [Scenario('ar1', context_length=120, horizon=8),
         Scenario('seasonal', context_length=120, horizon=8)]
report = local(Pipeline(NaiveForecaster()).test(suite, seeds=[10, 11]))

summary = report.summary
print(f"{summary['cases']} cases, {summary['diagnostics']} findings, {summary['failed_cases']} failed cases")
print(', '.join(f'{count} {status}' for status, count in summary['status_counts'].items()))
display(pd.DataFrame(report.checks))

4 cases, 16 findings, 0 failed cases
4 PASS, 0 FAIL, 4 INCONCLUSIVE, 8 UNSUPPORTED, 0 ERROR


,check,title,passed,failed
0,runs,Runs without errors,4,0
1,dates,Forecasts the requested dates,4,0
2,series,"Returns every series, in order",4,0
3,shape,Right number of steps and series,4,0
4,finite,No missing or infinite values,4,0


Every case passed every output check that ran. There is no quantiles row: the last-value forecaster gives no quantiles, so that check never ran. Each case has four findings; here are the first case's:

In [19]:
display(pd.DataFrame([{'finding': d.name, 'verdict': d.status, 'what it says': d.message}
                      for d in report.cases[0].diagnostics]))

,finding,verdict,what it says
0,forecast_contract,PASS,"Forecast indices, target order, shapes, finiteness, and quantile order satisfy the declared contract."
1,point_accuracy,INCONCLUSIVE,Point scores are descriptive; no capability acceptance tolerance was declared.
2,quantile_accuracy,UNSUPPORTED,No explicit quantile forecasts were supplied.
3,sample_scores,UNSUPPORTED,No joint predictive samples were supplied.


The output checks PASS. Point accuracy is INCONCLUSIVE: it was scored, but nobody set a pass mark. The quantile and sample scores are UNSUPPORTED: the forecast has neither, so there was nothing to check. Neither of those counts as a pass.

## Gate a build on it

`assert_passed()` is the line to put in CI. It raises on any FAIL or ERROR, that is on a broken output or a crash, and otherwise returns the report. We run three forecasters on the same cases: the last value, the AR(1) baseline, and a last-value model with an off-by-one bug that labels its forecast one step late.

In [20]:
class OneStepLate:
    """The last value, labeled one step late: an off-by-one bug."""

    def fit(self, context):
        self.last_ = context.iloc[-1]
        return self

    def predict(self, future_index):
        return pd.DataFrame([self.last_.to_numpy()] * len(future_index),
                            index=future_index + 1, columns=self.last_.index)

In [21]:
def passes_gate(report):
    try:
        report.assert_passed()
    except AssertionError:
        return False
    return True


def mean_error(report):
    errors = [case.metrics['mean_reference_mse'] for case in report.cases if case.metrics]
    return round(sum(errors) / len(errors), 2) if errors else 'not scored'


pipelines = {'Last value': Pipeline(NaiveForecaster()),
             'AR(1) baseline': Pipeline(GaussianAR1Forecaster(n_samples=64, seed=5)),
             'One step late': Pipeline(OneStepLate)}
reports = {name: local(pipeline.test(suite, seeds=[10, 11])) for name, pipeline in pipelines.items()}
gate = pd.DataFrame([{'forecaster': name, 'passes the gate': passes_gate(r),
                      'broken cases': r.summary['failed_cases'],
                      'error against the answer key': mean_error(r)}
                     for name, r in reports.items()]).set_index('forecaster')
assert gate['passes the gate'].tolist() == [True, True, False]  # weak or strong passes; wrong dates fail
gate

,passes the gate,broken cases,error against the answer key
forecaster,,,
Last value,True,0,1.21
AR(1) baseline,True,0,0.27
One step late,False,4,not scored


The off-by-one model fails the gate. This is what a CI log shows: each failed check in plain words, then every failing case with its seed.

In [22]:
try:
    reports['One step late'].assert_passed()
except AssertionError as failure:
    print(failure)

TimeGym detected failures:
Forecasts the requested dates: failed in 4 of 4 cases
  ar1[seed=10] contract_index_mismatch: Forecast index does not match the requested future index.
  ar1[seed=11] contract_index_mismatch: Forecast index does not match the requested future index.
  seasonal[seed=10] contract_index_mismatch: Forecast index does not match the requested future index.
  seasonal[seed=11] contract_index_mismatch: Forecast index does not match the requested future index.


The gate separates broken from working, not good from bad. The last value and the AR(1) baseline both pass, though the baseline's mean error against the answer key is under a quarter of the last value's (0.27 against 1.21). The broken forecasts get no score at all. TimeGym sets no pass mark for accuracy, because only you know what is good enough; if you need one, assert it on the cases' `metrics` yourself.

## Save it, and load it back

`to_json(path)` writes the report to a file and `TestReport.read(path)` loads it, with nothing lost. Loading is strict: it checks every field and recounts the summary, and it never runs code. We save the AR(1) report and read it back:

In [23]:
scratch = TemporaryDirectory(prefix='timegym-reports-')
report_path = Path(scratch.name) / 'report.json'
reports['AR(1) baseline'].to_json(report_path)
loaded = TestReport.read(report_path)
assert loaded.to_dict() == reports['AR(1) baseline'].to_dict()  # nothing lost through the file

Now four damaged copies, each written to a file and read back:

In [24]:
def read_damaged(source, edit):
    """Save a copy of a report with one edit, then try to read it back."""
    data = json.loads(source.to_json())
    edit(data)
    damaged_path = Path(scratch.name) / 'damaged.json'
    damaged_path.write_text(json.dumps(data))
    try:
        TestReport.read(damaged_path)
    except ValueError as refusal:
        return str(refusal)
    return 'loaded'


def first_case(data):
    return data['cases'][0]


refusals = pd.Series({
    'A FAIL edited to PASS': read_damaged(
        reports['One step late'], lambda d: first_case(d)['diagnostics'][0].update(status='PASS')),
    'A score set to NaN': read_damaged(loaded, lambda d: first_case(d)['metrics'].update(mse=float('nan'))),
    'A verdict TimeGym does not have': read_damaged(
        loaded, lambda d: first_case(d)['diagnostics'][0].update(status='MAYBE')),
    'An extra field (a stored forecast)': read_damaged(loaded, lambda d: first_case(d).update(forecast=[1.0, 2.0])),
}, name='what read() says')
refusals.index.name = 'what we changed in the file'
assert (refusals != 'loaded').all()  # every damaged copy is refused
refusals.to_frame()

,what read() says
what we changed in the file,
A FAIL edited to PASS,Stored report.summary disagrees with the derived diagnostic counts
A score set to NaN,Invalid report JSON: Nonfinite JSON constant is not permitted: NaN
A verdict TimeGym does not have,Unsupported diagnostic status
An extra field (a stored forecast),"report.cases[0] must contain exactly these fields: diagnostics, manifest, metrics, scenario, seed"


Each damaged file is refused with a reason, instead of loading as a report that says something else. The first row matters most: a failure quietly edited into a pass no longer matches the saved summary. `TestReport.from_json(text)` and `TestReport.from_dict(data)` apply the same checks to text and to parsed data.

## What the report records about the run

To rerun a case you need its recipe and the run's settings. The case's manifest holds the recipe; the report's metadata holds the model settings, the check settings and the environment.

In [25]:
meta = loaded.metadata
recipe = loaded.cases[0].manifest
model = meta['pipeline_config']['model']
checks = meta['diagnostic_settings']
display(pd.DataFrame({'recorded': {
    'Series recipe': f"{recipe['kind']}, {recipe['context_length']} steps of history, "
                     f"{recipe['horizon']} to forecast, seed {recipe['seed']}",
    'Model': f"{model['class']} with {model['parameters']}",
    'Replayable from the file': meta['pipeline_config']['replayable'],
    'Check settings': f"joint-spread check {'on' if checks['check_dependence'] else 'off'}",
    'Python': meta['python'],
    'Libraries': ', '.join(f'{name} {version}' for name, version in meta['dependencies'].items()),
    'Source fingerprint': f"{meta['source']['algorithm']} {meta['source']['digest'][:16]}..., "
                          f"{meta['source']['file_count']} files",
}}))
assert runtime_provenance()['source'] == meta['source']  # this kernel computes the same fingerprint

,recorded
Series recipe,"ar1, 120 steps of history, 8 to forecast, seed 10"
Model,"GaussianAR1Forecaster with {'levels': [0.1, 0.5, 0.9], 'n_samples': 64, 'seed': 5}"
Replayable from the file,True
Check settings,joint-spread check off
Python,3.13.18
Libraries,"numpy 2.5.4, pandas 3.1.1, scikit-learn 1.9.1, scipy 1.18.1, timegym 1.1.0"
Source fingerprint,"sha256 5174f6ac8188eba6..., 25 files"


This kernel computes the same fingerprint as the saved report, because both hash the installed package. That also means an edit in a notebook counts only after you save it, export it and restart the kernel. The report does not store the forecasts: replay regenerates them from the recipe and the model settings.

## Which runs can be replayed from the file alone?

Replay has to rebuild the model from the report. It does that only for TimeGym's own models with fixed settings, and it never imports or runs code named in a file. For anything else the report says why:

In [26]:
def naive_factory():
    return NaiveForecaster()


candidates = {
    'Last value': Pipeline(NaiveForecaster()),
    'AR(1) baseline, seed 5': Pipeline(GaussianAR1Forecaster(n_samples=64, seed=5)),
    'AR(1) baseline, no seed': Pipeline(GaussianAR1Forecaster(seed=None)),
    'Your own class (OneStepLate)': Pipeline(OneStepLate),
    'A factory function': Pipeline(model_factory=naive_factory),
}
configs = {name: pipeline_config(pipeline) for name, pipeline in candidates.items()}
display(pd.DataFrame([{'pipeline': name, 'replayable': config['replayable'],
                       'why not': config['model'].get('reason', '')}
                      for name, config in configs.items()]).set_index('pipeline'))

rebuilt = pipeline_from_config(configs['AR(1) baseline, seed 5'])['model']
assert rebuilt.params() == candidates['AR(1) baseline, seed 5'].model.params()  # the same settings come back
try:
    pipeline_from_config(configs['Your own class (OneStepLate)'])
except ValueError as refusal:
    print(f'Rebuilding your own class: {refusal}')

,replayable,why not
pipeline,,
Last value,True,
"AR(1) baseline, seed 5",True,
"AR(1) baseline, no seed",False,Exact replay requires a fixed nonnegative integer model seed.
Your own class (OneStepLate),False,Only exact bundled native baseline classes support automatic replay.
A factory function,False,Factory execution requires explicitly supplied user code.


Rebuilding your own class: This model configuration is not automatically replayable; supply its original model code


Without a seed, the AR(1) baseline draws different samples on every run, so an exact replay is impossible and the report says so. For a model you wrote, keep the code that defines it next to the report, with its settings and seeds.

## Replay a case months later

`replay_case` reads the recipe and the settings from the report, rebuilds the model, regenerates the series, reruns every check, and compares. Here we replay the first case from the saved file, as you would in a new session. `replay_case` belongs to the installed package, so we read the file with the installed `TestReport`; the file is the same either way.

In [27]:
replayed = InstalledTestReport.read(report_path).replay_case(0)
comparison = replayed.metadata['replay']
questions = {'metrics_match': 'Same scores', 'diagnostics_match': 'Same verdicts and evidence',
             'manifest_match': 'Same series recipe', 'traceback_match': 'Same error tracebacks',
             'source_matches': 'Same TimeGym source', 'dependencies_match': 'Same library versions',
             'python_matches': 'Same Python version', 'reproduced': 'Reproduced',
             'same_environment': 'Same environment'}
assert comparison['reproduced'] and comparison['same_environment']  # same machine: an exact match
pd.DataFrame({'answer': {question: comparison[key] for key, question in questions.items()}})

,answer
Same scores,True
Same verdicts and evidence,True
Same series recipe,True
Same error tracebacks,True
Same TimeGym source,True
Same library versions,True
Same Python version,True
Reproduced,True
Same environment,True


**Reproduced** means the scores, the verdicts and the recipe all came out the same. **Same environment** means the source fingerprint, the library versions and the Python version match too. They are separate answers because a file travels. Below we pretend this one was saved on a machine with an older NumPy:

In [28]:
saved = json.loads(report_path.read_text())
saved['metadata']['dependencies']['numpy'] = '1.26.4'  # pretend the file came from an older NumPy
elsewhere = InstalledTestReport.from_dict(saved).replay_case(0).metadata['replay']
scratch.cleanup()
assert elsewhere['reproduced'] and not elsewhere['same_environment']  # same results, different environment
pd.DataFrame({'answer': {questions[key]: elsewhere[key]
                         for key in ('reproduced', 'dependencies_match', 'same_environment')}})

,answer
Reproduced,True
Same library versions,False
Same environment,False


Replay still runs and the results still match, and the comparison says the environment differs. On a really different stack, small floating-point differences can make **Same scores** false even when the code is unchanged, so read the environment rows first. From a shell, `python examples/reproduce_report.py report.json` prints the same comparison and exits with status 1 when the case is not reproduced; add `--require-same-environment` to also fail on a different environment.

**Limits.** A PASS covers the check that ran and nothing more. Passing every output check says a forecast is well formed, not that it is accurate, and an UNSUPPORTED check proves nothing either way. Replay rebuilds only TimeGym's own models with fixed settings. The source fingerprint covers the installed `timegym` files, not your model's code. Loading checks that a report is consistent, not who wrote it: a pass edited in together with a matching summary would load.

## Change one thing

- Drop the `+ 1` in `OneStepLate`: it passes the gate and scores exactly like the last value, because now it is the last value.
- In the last cell, change the saved model's `seed` from 5 to 6 instead of the NumPy version: **Reproduced** turns false (and the assert stops you), because the rebuilt model draws different samples.
- Give the AR(1) baseline `target_transform=DifferenceTransform()` from `timegym.target`: `pipeline_config` still marks it replayable, because the transform is built in too.

**Next:** [11 · Plotting](11_plotting.ipynb) draws the forecasts and errors behind a report. [Faults, repairs and replay](23_faults_repairs_and_replay.ipynb) uses reports to catch real bugs and replay them.